# AGMS-Net Phase 2: Regime-Aware Gating (v3 — Scale-Invariant Features)

**What changed from v1/v2:**

| Version | Features | Problem | Status |
|---------|----------|---------|--------|
| v1 | μ, σ, trend, kurtosis, autocorr | Unconstrained addition → MSE↓ but MAE↑ | ❌ |
| v2 | Same 5 + learnable scale (init=0.01) | Scale grew to 0.08–0.11 but divergence unchanged | ❌ |
| **v3** | **norm_trend, kurtosis, autocorr only** | Removes absolute-scale features (μ, σ) that caused mean-shift | ✅ |

**Root cause of v1/v2 failure:** Raw μ and σ encoded absolute channel levels. After RevIN inverse, this created a systematic mean-shift: fewer large errors (MSE↓) but consistent offset on every prediction (MAE↑).

**v3 fix:** 3 scale-invariant features only — none can encode absolute signal level:
1. **Normalized trend** = (last − first) / σ → direction + speed, unit-free
2. **Kurtosis** = E[(x−μ)⁴/σ⁴] → spiky burst vs flat plateau
3. **Autocorr lag-1** = corr(xₜ, xₜ₋₁) → periodic vs noisy

**Runtime:** GPU required (Runtime > Change runtime type > T4 GPU)

**Dataset:** Same `alibaba.csv` (96 machines, 11206 timesteps)

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Clone Repository (Phase 2 v3 code)

In [ ]:
import os
!rm -rf /content/Workload_Prediction_MTP
!git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('run.py exists:', os.path.exists('run.py'))

with open('models/MSCNet.py') as f:
    content = f.read()
    assert 'RegimeDetector' in content, 'ERROR: RegimeDetector not found!'
    assert 'AdaptiveGate' not in content, 'ERROR: AdaptiveGate still present!'
    assert 'regime_scale' in content, 'ERROR: regime_scale not found!'
    assert 'norm_trend' in content, 'ERROR: norm_trend not found — not v3 code!'
    assert 'nn.Linear(3, 32)' in content, 'ERROR: Expected 3-feature input, found mismatch!'
    assert 'nn.Linear(5, 32)' not in content, 'ERROR: Old 5-feature version still present!'
    print('Phase 2 v3 confirmed:')
    print('  - RegimeDetector with 3 scale-invariant features (norm_trend, kurtosis, autocorr)')
    print('  - regime_scale (learnable scalar injection control)')
    print('  - MSPBlock at Phase 0 baseline (static mapping)')

## 3. Install Dependencies

In [ ]:
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

## 4. Upload Dataset

In [ ]:
import os, pandas as pd
%cd /content
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/alibaba.csv'):
    from google.colab import files
    print('Upload alibaba.csv (96 machines, ~11MB):')
    uploaded = files.upload()
    for f in uploaded: os.rename(f, 'dataset/alibaba.csv')
df = pd.read_csv('dataset/alibaba.csv')
n_features = len(df.columns) - 1
print(f'Dataset: {df.shape}, Features: {n_features}')
print(f'Time: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')

## 5. Setup Paths & Symlinks

In [ ]:
import os
%cd /content
DATA_DIR  = '/content/dataset'
WORK_DIR  = '/content/Workload_Prediction_MTP/MSCNet_baseline'
os.makedirs(f'{WORK_DIR}/dataset', exist_ok=True)
if not os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'):
    os.symlink(f'{DATA_DIR}/alibaba.csv', f'{WORK_DIR}/dataset/alibaba.csv')
print('Symlink created:', os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'))
%cd {WORK_DIR}
print('Working directory:', os.getcwd())

## 6. Train All Horizons (H=24, 48, 72, 96)

Each takes ~8-12 minutes on T4. Total ~35-45 min.

In [ ]:
import os, subprocess, sys

HORIZONS = [24, 48, 72, 96]
N_FEATURES = 96
SEQ_LEN = 96

for h in HORIZONS:
    print(f'\n{"="*80}')
    print(f'  Training H={h}')
    print(f'{"="*80}\n')

    cmd = [
        sys.executable, 'run.py',
        '--task_name', 'long_term_forecast',
        '--is_training', '1',
        '--root_path', './dataset/',
        '--data_path', 'alibaba.csv',
        '--model_id', f'alibaba_{SEQ_LEN}_{h}',
        '--model', 'MSCNet',
        '--data', 'custom',
        '--features', 'M',
        '--seq_len', str(SEQ_LEN),
        '--label_len', str(SEQ_LEN),
        '--pred_len', str(h),
        '--e_layers', '3',
        '--d_layers', '1',
        '--factor', '3',
        '--enc_in', str(N_FEATURES),
        '--dec_in', str(N_FEATURES),
        '--c_out', str(N_FEATURES),
        '--d_model', '128',
        '--d_ff', '2048',
        '--n_heads', '4',
        '--des', 'Exp',
        '--itr', '1',
        '--batch_size', '32',
        '--learning_rate', '0.0001',
        '--train_epochs', '100',
        '--patience', '3',
        '--channel', str(N_FEATURES),
        '--seed', '2022',
    ]

    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        print(f'ERROR: H={h} failed with return code {result.returncode}')

## 7. Collect & Compare Results

In [ ]:
import os, re, numpy as np

phase0 = {24: (0.4630, 0.4927), 48: (0.5638, 0.5438), 72: (0.6447, 0.5794), 96: (0.7213, 0.6144)}
paper  = {24: 0.5687, 48: 0.6274, 72: 0.6622, 96: 0.6893}
phase2v1 = {24: (0.4959, 0.4640), 48: (0.5435, 0.5624), 72: (0.5787, 0.6381), 96: (0.6102, 0.7065)}

phase2v3 = {}
print('--- Raw Results Log ---')
for root, dirs, files in os.walk('results'):
    for f in files:
        if f == 'metrics.npy':
            path = os.path.join(root, f)
            m = np.load(path)
            folder = os.path.basename(root)
            print(f'{folder}')
            print(f'mse:{m[0]}, mae:{m[1]}, rmse:{m[2]}')
            match = re.search(r'alibaba_96_(\d+)_', folder)
            if match:
                h = int(match.group(1))
                phase2v3[h] = (float(m[0]), float(m[1]))
            print()

print()
print('=' * 110)
print('AGMS-Net Phase 2 v3 (3 Scale-Invariant Features) vs Phase 0 Baseline vs Paper (Alibaba, L=96)')
print('=' * 110)
hdr = f'{"H":>4} | {"P2v3 MSE":>10} | {"Phase0 MSE":>10} | {"vs P0":>7} | {"Paper MSE":>10} | {"P2v3 MAE":>10} | {"Phase0 MAE":>10} | {"vs P0":>7}'
print(hdr)
print('-' * 110)
for h in [24, 48, 72, 96]:
    if h in phase2v3:
        p_mse, p_mae = phase2v3[h]
        p0_mse, p0_mae = phase0[h]
        d_mse = (p_mse - p0_mse) / p0_mse * 100
        d_mae = (p_mae - p0_mae) / p0_mae * 100
        print(f'{h:4d} | {p_mse:10.4f} | {p0_mse:10.4f} | {d_mse:+6.1f}% | {paper[h]:10.4f} | {p_mae:10.4f} | {p0_mae:10.4f} | {d_mae:+6.1f}%')
print('=' * 110)

# Check if MSE-MAE divergence is fixed
print()
print('--- Divergence check (MSE and MAE should move in the SAME direction) ---')
for h in [24, 48, 72, 96]:
    if h in phase2v3:
        p_mse, p_mae = phase2v3[h]
        p0_mse, p0_mae = phase0[h]
        d_mse = (p_mse - p0_mse) / p0_mse * 100
        d_mae = (p_mae - p0_mae) / p0_mae * 100
        same_dir = (d_mse < 0 and d_mae < 0) or (d_mse > 0 and d_mae > 0)
        status = 'OK' if same_dir else 'DIVERGED'
        print(f'  H={h}: MSE {d_mse:+.1f}%, MAE {d_mae:+.1f}% -> {status}')

## 8. Regime Scale & Feature Analysis

In [ ]:
import torch, os, glob

checkpoint_dirs = sorted(glob.glob('checkpoints/long_term_forecast_alibaba_*'))
print(f'Found {len(checkpoint_dirs)} checkpoint(s)')

for ckpt_dir in checkpoint_dirs:
    ckpt_path = os.path.join(ckpt_dir, 'checkpoint.pth')
    if not os.path.exists(ckpt_path): continue
    print(f'\n--- {ckpt_path} ---')
    state = torch.load(ckpt_path, map_location='cpu')

    if 'regime_scale' in state:
        print(f'  regime_scale = {state["regime_scale"].item():.6f}  (init was 0.01)')
    regime_params = {k: v for k, v in state.items() if 'regime' in k}
    for k, v in regime_params.items():
        if k == 'regime_scale': continue
        print(f'  {k}: mean={v.mean():.4f}, std={v.std():.4f}')
    total = sum(v.numel() for v in state.values())
    rp = sum(v.numel() for v in regime_params.values())
    print(f'  Regime params: {rp:,} / {total:,} ({rp/total*100:.2f}%)')

## 9. Model Complexity

In [ ]:
import torch, sys, os
os.chdir('/content/Workload_Prediction_MTP/MSCNet_baseline')
sys.path.insert(0, '.')
from ptflops import get_model_complexity_info
from types import SimpleNamespace
configs = SimpleNamespace(
    task_name='long_term_forecast', seq_len=96, pred_len=24,
    enc_in=96, dec_in=96, c_out=96, d_model=128, d_ff=2048,
    n_heads=4, e_layers=3, d_layers=1, factor=3, dropout=0.05,
    output_attention=False, activation='gelu', moving_avg=25, channel=96
)
from models.MSCNet import Model
model = Model(configs)
macs, params = get_model_complexity_info(model, (96, 96), as_strings=True, print_per_layer_stat=False, verbose=False)
print(f'MACs: {macs}')
print(f'Parameters: {params}')